In [1]:
import ctypes
import numpy as np
import pandas as pd
import os
import plotly as plt 
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path
from torchvision import transforms
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

In [2]:
# === Charger la DLL Rust ===
lib = ctypes.CDLL("target/release/mymodel.dll")

In [3]:
lib.create_mlp_model.argtypes = [ctypes.c_size_t, ctypes.c_size_t, ctypes.c_double, ctypes.c_size_t,ctypes.c_bool,ctypes.c_bool]
lib.create_mlp_model.restype = ctypes.c_void_p

lib.train_mlp_model.argtypes = [
    ctypes.c_void_p,
    ctypes.POINTER(ctypes.c_double),
    ctypes.POINTER(ctypes.c_double),
    ctypes.c_size_t, ctypes.c_size_t
]
lib.train_mlp_model.restype = None

lib.predict_mlp_model.argtypes = [ctypes.c_void_p, ctypes.POINTER(ctypes.c_double), ctypes.c_size_t]
lib.predict_mlp_model.restype = ctypes.c_double

lib.create_mlp_classifier.argtypes = [
    ctypes.c_size_t,  # n_inputs
    ctypes.c_size_t,  # n_hidden
    ctypes.c_size_t,  # n_classes
    ctypes.c_double,  # learning_rate
    ctypes.c_size_t   # epochs
]
lib.create_mlp_classifier.restype = ctypes.c_void_p
lib.train_mlp_classifier.argtypes = [
    ctypes.c_void_p,                   # model_ptr
    ctypes.POINTER(ctypes.c_double),  # x_ptr
    ctypes.POINTER(ctypes.c_uint32),  # y_ptr
    ctypes.c_size_t,                  # n_samples
    ctypes.c_size_t                   # n_features
]
lib.train_mlp_classifier.restype = None

lib.predict_mlp_classifier.argtypes = [
    ctypes.c_void_p,
    ctypes.POINTER(ctypes.c_double),
    ctypes.c_size_t
]
lib.predict_mlp_classifier.restype = ctypes.c_uint32

In [4]:
# === Paramètres ===
IMAGE_SIZE = (64, 64)
SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg"}
FILTER_LABELS = ["A", "B", "C"]  

# === Transformer torchvision ===
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.Grayscale(),
    transforms.ToTensor()
])

# === Chargement des images ===
def load_images_from_directory(root_dir, filter_labels=None):
    images, labels = [], []
    root_dir = Path(root_dir)

    for label_dir in sorted(root_dir.iterdir()):
        if not label_dir.is_dir():
            continue
        label = label_dir.name
        if filter_labels and label not in filter_labels:
            continue

        for img_path in label_dir.glob("*"):
            if img_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
                continue
            try:
                img = Image.open(img_path).convert("RGB")
                images.append(img)
                labels.append(label)
            except Exception as e:
                print(f"Erreur {img_path}: {e}")
    
    return images, labels



In [5]:
# === Prétraitement pour MLP classifier (X: float64, y: uint32) ===
def preprocess_for_mlp(images, labels):
    X = np.stack([transform(img).numpy().flatten() for img in images])
    X = X.astype(np.float64)

    le = LabelEncoder()
    y_int = le.fit_transform(labels).astype(np.uint32)

    return X, y_int, le

In [6]:
# === Charger et préparer les données ===
train_images, train_labels = load_images_from_directory("dataset/train", FILTER_LABELS)
test_images, test_labels = load_images_from_directory("dataset/test", FILTER_LABELS)

X_train, y_train, label_encoder = preprocess_for_mlp(train_images, train_labels)
X_test, y_test, _ = preprocess_for_mlp(test_images, test_labels)

n_samples, n_features = X_train.shape
n_classes = len(np.unique(y_train))


In [ ]:
# === Créer le modèle MLP ===
hidden_units = 100
learning_rate = 0.01
epochs = 3000

model_ptr = lib.create_mlp_classifier(n_features, hidden_units, n_classes, learning_rate, epochs)

# === Entraînement ===
x_ptr = X_train.flatten().ctypes.data_as(ctypes.POINTER(ctypes.c_double))
y_ptr = y_train.ctypes.data_as(ctypes.POINTER(ctypes.c_uint32))

lib.train_mlp_classifier(model_ptr, x_ptr, y_ptr, n_samples, n_features)


In [22]:
# === Prédictions ===
y_pred = []
for x in X_test:
    x_ptr = x.astype(np.float64).ctypes.data_as(ctypes.POINTER(ctypes.c_double))
    pred = lib.predict_mlp_classifier(model_ptr, x_ptr, n_features)
    y_pred.append(pred)

y_pred = np.array(y_pred)

# === Évaluation ===
acc = accuracy_score(y_test, y_pred)
print(f"✅ Accuracy MLP : {acc * 100:.2f}%")

✅ Accuracy MLP : 33.33%


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

clf = LogisticRegression(max_iter=3000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("Baseline sklearn logistic regression:", accuracy_score(y_test, y_pred))


Répartition des prédictions : Counter({2: 3})
